# Serve the Marma translator from Google Colab

Runs the FastAPI backend (`backend/`) on a Colab GPU and exposes it through an ngrok tunnel.

**Before running:**
1. Copy the contents of `backend/` into `MyDrive/Marma_Backend/` on Google Drive.
2. Put the fine-tuned model folder on Drive and set `MODEL_PATH` below.
3. Add your ngrok auth token as a Colab secret named `NGROK_AUTH_TOKEN` (key icon in the left sidebar). Never paste it into a cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install dependencies
!pip install fastapi uvicorn pydantic pyngrok nest-asyncio
!pip install torch transformers accelerate bitsandbytes

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import os

# 1. Set your model path here
os.environ["MODEL_PATH"] = "/content/drive/MyDrive/Marma_ImperativeFinal_Model_Final"

# 2. Setup Ngrok (token comes from Colab Secrets)
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")  # stored in Colab Secrets
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print("="*60)
print(f"🌟 YOUR API IS LIVE AT: {public_url}")
print("Set window.MARMA_API_URL in frontend/config.js to:")
print(f"window.MARMA_API_URL = '{public_url}/translate';")
print("="*60)

# 3. Change directory to where your python files are
%cd /content/drive/MyDrive/Marma_Backend



In [ ]:
# Install Unsloth and its core dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers peft accelerate bitsandbytes

In [ ]:
# 4. Run the FastAPI app
import asyncio

# Replace your previous uvicorn.run() line with this:
config = uvicorn.Config("main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# This starts the server within the existing Colab event loop
await server.serve()